In [4]:
from langgraph.graph import StateGraph, START, END
from typing import Dict, Optional
from pydantic import BaseModel

In [5]:
class BatsmanState(BaseModel):

    runs: int
    balls: int
    fours: int 
    sixes: int 

    strike_rate: Optional[float] = None
    boundary_per_ball: Optional[float] = None
    boundary_percent: Optional[float] = None
    summary: Optional[str] = None

In [6]:
def calculate_strike_rate(state: BatsmanState) -> Dict[str, any]:
    runs = state.runs
    balls = state.balls
    strike_rate = runs/balls
    return {'strike_rate': strike_rate}

def calculate_boundary_percent(state: BatsmanState) -> Dict[str, any]:
    boundary_runs = state.fours*4 + state.sixes*6
    runs = state.runs
    boundary_percent = (boundary_runs / runs)*100
    return {'boundary_percent': boundary_percent}

def calculate_boundary_per_ball(state: BatsmanState) -> Dict[str, any]:
    boundary = state.fours + state.sixes
    balls = state.balls
    boundary_per_ball = boundary/balls
    return {'boundary_per_ball': boundary_per_ball}

In [7]:
def summary(state: BatsmanState):
    summary = f"""
Strike Rate - {state.strike_rate}
Balls per boundary - {state.boundary_per_ball}
Boundary percent - {state.boundary_percent}
"""
    return {'summary': summary}

In [11]:
graph = StateGraph(BatsmanState)

graph.add_node('calculate_strike_rate', calculate_strike_rate)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('calculate_boundary_per_ball', calculate_boundary_per_ball)
graph.add_node('summary', summary)

graph.add_edge(START, 'calculate_strike_rate')
graph.add_edge(START, 'calculate_boundary_percent')
graph.add_edge(START, 'calculate_boundary_per_ball')

graph.add_edge('calculate_strike_rate', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')
graph.add_edge('calculate_boundary_per_ball', 'summary')
graph.add_edge('summary', END)


workflow = graph.compile()

In [13]:
initial_state = {'runs': 100,
    'balls': 50,
    'fours': 10,
    'sixes': 5 }
workflow.invoke(initial_state)

{'runs': 100,
 'balls': 50,
 'fours': 10,
 'sixes': 5,
 'strike_rate': 2.0,
 'boundary_per_ball': 0.3,
 'boundary_percent': 70.0,
 'summary': '\nStrike Rate - 2.0\nBalls per boundary - 0.3\nBoundary percent - 70.0\n'}